In [ ]:
# Ollama default port: 11434
# ssh -L 11434:localhost:11434 USERNAME@HOST

### Imports

In [1]:
import pandas as pd
import os
from testgen.utils import *
from pathlib import Path
from dotenv import load_dotenv

In [2]:
base_path = Path().cwd()
load_dotenv(override=True)

True

In [3]:
# read data and rename columns
df = pd.read_excel(base_path / "data/sensor_requirements.xlsx")
df_examples = pd.read_excel(base_path / "data/sensor_examples.xlsx")

### Drop Sensor

In [4]:
from testgen.prompts import Sensors

# Acc: Acceleration Pedal
# WSA: Wheel Steering Angle
# WS: Wheel Speed
# YR: Yaw Rate
# ST: Steering Torque
target_sensor = ["Wheel Steering Angle", "Steering Torque"]

Sensors = Sensors.split("\n")
# drop all sensors in Sensors
Sensors = "\n".join(filter(lambda s: all([s.find(f"{t} (") == -1 for t in target_sensor]), Sensors))

print(Sensors)

Acceleration Pedal (Acc): Measures the amount of pressure applied to the accelerator pedal, indicating the driver's desired acceleration.
Wheel Speed (WS): Measures the rotational speed of the vehicle's wheels, providing information on the vehicle's speed and potential wheel slippage.
Yaw Rate (YR): Measures the rate of rotation around the vertical axis of the vehicle, indicating its turning behavior and stability.


In [5]:
# delete targeted sensors
df = df.drop(columns=[c.replace(" ", "_").lower() for c in target_sensor])
df_examples = df_examples.drop(columns=[c.replace(" ", "_").lower() for c in target_sensor])

### Find Examples

In [65]:
def get_example_txt(df_examples, N_EXAMPLES=1):

    if N_EXAMPLES > 1:
        indexes_to_drop, examples = get_examples_from_df(
            df_examples.drop(columns=["selected"]), N_EXAMPLES
        )

    if N_EXAMPLES == 1:  # pick pre selected example when N_EXAMPLES = 1
        df_examples_n1 = df_examples[df_examples["selected"] == 1].copy()
        df_examples_n1.drop(columns=["selected"], inplace=True)
        indexes_to_drop, examples = get_examples_from_df(df_examples_n1, 1)

    # join all examples in a text format to add to prompt
    examples_txt = ""

    for e1 in examples.values():
        for e2 in e1:
            examples_txt += f"Requirement: {e2[0]}\n"
            # examples_txt += f"Vector: {e2[1]}\n"
            examples_txt += f"Target Sensor/s: {e2[1]}\n"
            examples_txt += "\n"

    return examples_txt, examples

In [66]:
examples_txt, examples = get_example_txt(df_examples, 1)

print("\n".join(examples_txt.split("\n")[-10:]))

Requirement: The control system must detect and mitigate power oversteer to maintain vehicle control by adjusting the wheel steering angle and acceleration pedal inputs
Target Sensor/s: [acceleration_pedal]

Requirement: The system must rapidly process wheel speed information to activate anti-lock braking systems (ABS) and prevent wheel lockup during emergency braking scenarios
Target Sensor/s: [wheel_speed]

Requirement: Drift control systems must adaptively use yaw rate feedback to optimize vehicle control during high-performance driving scenarios
Target Sensor/s: [yaw_rate]




In [67]:
Negative_N_EXAMPLES = 2

if Negative_N_EXAMPLES > 0:
    df_examples_t = df_examples.drop(columns=["selected"])
    negative_examples = df_examples_t[
        df_examples_t.iloc[:, 1:].sum(axis=1) == 0
    ].sample(Negative_N_EXAMPLES)
    negative_examples["vector"] = "[]"

    examples["negative"] = negative_examples[["requirement", "vector"]].values.tolist()


# add to example txt
for e_n in examples["negative"]:
    examples_txt += f"Requirement: {e_n[0]}\n"
    examples_txt += f"Target Sensor/s: {e_n[1]}\n"
    examples_txt += "\n"

print("\n".join(examples_txt.split("\n")[-10:]))

Requirement: Drift control systems must adaptively use yaw rate feedback to optimize vehicle control during high-performance driving scenarios
Target Sensor/s: [yaw_rate]

Requirement: The power steering system must adapt the torque levels in response to detected road surface conditions (e.g., ice, water, gravel)
Target Sensor/s: []

Requirement: The system must detect and report any failures or errors of the steering angle sensor within a specified time frame
Target Sensor/s: []




# Response Format

In [72]:
from pydantic import Field, create_model

Sensors_t = Sensors.split("\n")


def clean_sensors_fn(x):
    x = x.split(":")
    x[0] = x[0][: x[0].find("(")]
    return x


Sensors_t = list(map(clean_sensors_fn, Sensors_t))

sensor_attrs = {}

for sensor in Sensors_t:
    sensor_attrs[sensor[0].strip().lower().replace(" ", "_")] = (
        int,
        Field(description=sensor[1].strip()),
    )


VectorFormat = create_model("VectorFormat", **sensor_attrs)

# LLM

In [73]:
from testgen.prompts import SystemPrompt
from testgen.prompts import UserPrompt

In [ ]:
llm_models = {
    "ollama": ["phi4:latest"],
}

endpoint_attrs = {
    "ollama": {
        "api_key": os.getenv("OLLAMA_API_KEY"),
        "base_url": os.getenv("OLLAMA_ENDPOINT"),
    }
}

# Run for all Requirements

In [ ]:
for n_example in  [1, 3, 5, 8]:

    examples_txt, examples = get_example_txt(df_examples, n_example)

    for endpoint_name in llm_models.keys():

        for model_name in llm_models[endpoint_name]:

            print(f"Running {model_name} on {endpoint_name} examples {n_example}...")

            client = llm_client(endpoint_name, **endpoint_attrs[endpoint_name])

            results = client_invoke_sensor(
                endpoint_name,
                model_name,
                client,
                df,
                SystemPrompt,
                Sensors,
                examples_txt,
                UserPrompt,
                response_format=VectorFormat,
            )

            (
                number_of_requests,
                accuracy,
                avg_time_per_req,
                avg_token_per_req,
                avg_completion_token_per_req,
                total_tokens,
                total_completion_tokens,
                total_time,
            ) = calc_stats(results)

            results_file = save_responses(
                base_path,
                "sensor_dropped",
                model_name=model_name.split("/")[-1],
                n_examples=n_example,
                examples=examples,
                accuracy=accuracy,
                number_of_requests=number_of_requests,
                total_tokens=total_tokens,
                total_completion_tokens=total_completion_tokens,
                avg_token_per_req=avg_token_per_req,
                avg_completion_token_per_req=avg_completion_token_per_req,
                avg_time_per_req=avg_time_per_req,
                results=results,
            )

            print(f"Done")